# 🧮 逻辑回归：最简单的分类问题解决办法

湖北理工学院《人工智能理论》课程资料

📝 **本节内容概述：**
1. 🎲 **数据准备**：使用内置方法生成虚拟的分类数据，避免硬件压力。
2. ✌️ **二分类**：基于 PyTorch 和 Sigmoid 分类实现。
3. 🤘 **多分类详解**：分别演示 OvR、OvO、以及现代深度学习主流的 Softmax 多项分布回归模型。

## 🎲 1. 数据准备

为了更直观地展示分类边界和底层矩阵计算过程，我们使用 `scikit-learn` 生成二维的虚拟数据。这使我们能够在二维坐标系上清晰地看到模型是如何通过决策平面将特征空间切分的。

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from IPython.display import clear_output

# 全局画图设置字体，防止中文方块乱码
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 生成二分类数据 (样本: 500, 特征: 2维坐标)
X_bin, y_bin = make_classification(n_samples=500, n_features=2, n_redundant=0, n_informative=2, random_state=42, n_clusters_per_class=1)
X_bin_train, X_bin_test, y_bin_train, y_bin_test = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)

# 生成三分类（多分类）数据 (3个斑点聚集簇，同样在 2 维空间中)
X_multi, y_multi = make_blobs(n_samples=500, n_features=2, centers=3, cluster_std=1.5, random_state=42)
X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.scatter(X_bin[:, 0], X_bin[:, 1], c=y_bin, cmap=plt.cm.coolwarm, edgecolors='k')
ax1.set_title('二分类虚拟数据')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i in range(3):
    ax2.scatter(X_multi[y_multi == i, 0], X_multi[y_multi == i, 1], label=f'类别 {i}', color=colors[i], edgecolors='k')
ax2.set_title('三分类虚拟数据')
ax2.legend()
plt.show()

## ✌️ 2. 基于逻辑回归的二分类

二分类逻辑回归本质也是一个线性模型。但为了处理分类问题，我们将线性的输出喂进一个 **Sigmoid 激活函数**，将其映射到 $(0, 1)$ 之间，把这个结果视作“样本属于正例的概率”。此部分使用 **二元交叉熵 (BCE)** 损失计算。

In [ ]:
# 1. 数组维度整理与 PyTorch 张量转换
X_train_t = torch.tensor(X_bin_train, dtype=torch.float32)
y_train_t = torch.tensor(y_bin_train, dtype=torch.float32).view(-1, 1) # 将标签形状设为列向量 (N, 1) 以匹配预测输出

# 2. 核心参数初始化
# w 是连接特征和输出的权重，形状为(2, 1)。requires_grad 让 PyTorch 监测它用于求导
w = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.5
epochs = 500

# 3. 准备网格坐标，为绘制覆盖全图的决策背景底色做准备
margin = 3 # 决策色块向四面八方边界扩充 3 的缓冲距离
x_min, x_max = X_bin[:, 0].min() - margin, X_bin[:, 0].max() + margin
y_min, y_max = X_bin[:, 1].min() - margin, X_bin[:, 1].max() + margin
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

loss_history = []

for epoch in range(epochs):
    # --- 前向传播：计算属于分类 1 的推断概率 ---
    y_pred = torch.sigmoid(torch.mm(X_train_t, w) + b)
    
    # --- 误差函数计算 ---
    # 根据二元交叉熵公式，正确标签若为 1 则激活第一项 -log(P)，为 0 激活第二项 -log(1-P)
    loss = -torch.mean(y_train_t * torch.log(y_pred + 1e-8) + (1 - y_train_t) * torch.log(1 - y_pred + 1e-8))
    
    # --- 反向传播 ---
    loss.backward()
    
    # --- 参数更新 ---
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()  # 切记：防止每一轮反向传播梯度累加残存
        b.grad.zero_()
        
    loss_history.append(loss.item())
    
    # === 动画绘制环节 (对底层算法无影响，仅服务学习体验) ===
    if (epoch + 1) % 50 == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(loss_history, 'b-')
        ax1.set_title(f'二分类训练损失 (Epoch {epoch+1})')
        
        with torch.no_grad():
            Z = torch.sigmoid(torch.mm(grid_tensor, w) + b)
            Z = (Z >= 0.5).float().numpy().reshape(xx.shape)
            
        ax2.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
        ax2.scatter(X_bin_train[:, 0], X_bin_train[:, 1], c=y_bin_train, cmap=plt.cm.coolwarm, edgecolors='k')
        ax2.set_xlim(x_min, x_max)
        ax2.set_ylim(y_min, y_max)
        ax2.set_title('二分类决策边界')
        plt.show()

## 🤘 3. 逻辑回归的多分类拓展

当我们面临的情境从“二选一”变成“多选一”时，逻辑回归有三种主流的处理策略：
1. **一对多 (One-vs-Rest, OvR)**
2. **一对一 (One-vs-One, OvO)**
3. **Softmax 函数 (多项分布模型)**

接下来，我们使用独立的三个代码单元。带你一行一行手写这些分类器的底层操作，以深入理解矩阵在这个过程中的流向。

### 3.1 一对多 (One-vs-Rest, OvR)

**核心思想**：对于 $K$ 个类别，我们训练 $K$ 个二分类器。第 $i$ 个分类器负责专门判断“样本是否属于第 $i$ 类”。
**预测机制**：让 $K$ 个模型对一个样本独立预测。它们各自计算出属于自己防区的概率。**谁的概率得分排第一（最大值 Argmax），判断结论就听谁的。**

In [ ]:
# 1. 数组维度整理与张量抓取
X_m_train_t = torch.tensor(X_multi_train, dtype=torch.float32)
y_m_train_t = torch.tensor(y_multi_train, dtype=torch.long)

# 2. 既然要训练三个二分类器，权重矩阵不妨并排在一起构成形状为 (2, 3) 的大矩阵 (3列负责对应1支分类器)
W_ovr = torch.randn(2, 3, requires_grad=True)
b_ovr = torch.zeros(3, requires_grad=True)

# 将数字形式标签 [0, 1, 2] 转换为 One-Hot (独热) 形式方便一次性比对误差。 (例：1 -> [0, 1, 0])
y_onehot = torch.nn.functional.one_hot(y_m_train_t, num_classes=3).float()

margin = 3
x_min, x_max = X_multi[:, 0].min() - margin, X_multi[:, 0].max() + margin
y_min, y_max = X_multi[:, 1].min() - margin, X_multi[:, 1].max() + margin
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

loss_history_ovr = []
from matplotlib.colors import ListedColormap
custom_cmap = ListedColormap(['#1f77b4', '#ff7f0e', '#2ca02c'])

for epoch in range(epochs):
    # 三个分类器同时启动：得到形状 (N, 3) 概率表。每一列代表对应模型判定为真例的 Sigmoid 概率
    y_pred = torch.sigmoid(torch.mm(X_m_train_t, W_ovr) + b_ovr)
    
    # 巧妙利用 One-Hot 计算3个分类器独立的平均分类交叉熵
    loss = -torch.mean(y_onehot * torch.log(y_pred + 1e-8) + (1 - y_onehot) * torch.log(1 - y_pred + 1e-8))
    
    # 反向更新流程不变
    loss.backward()
    with torch.no_grad():
        W_ovr -= learning_rate * W_ovr.grad
        b_ovr -= learning_rate * b_ovr.grad
        W_ovr.grad.zero_()
        b_ovr.grad.zero_()
    loss_history_ovr.append(loss.item())

    if (epoch + 1) % 50 == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(loss_history_ovr, 'm-')
        ax1.set_title(f'OvR 整体训练损失 (Epoch {epoch+1})')
        
        with torch.no_grad():
            grid_pred = torch.sigmoid(torch.mm(grid_tensor, W_ovr) + b_ovr)
            # OvR 的精髓：比较它们，取其中呼声 (概率分数) 最高的作为类别下标
            Z = torch.argmax(grid_pred, dim=1).numpy().reshape(xx.shape)
            
        ax2.contourf(xx, yy, Z, cmap=custom_cmap, alpha=0.3)
        ax2.scatter(X_m_train_t[:, 0], X_m_train_t[:, 1], c=y_m_train_t, cmap=custom_cmap, edgecolors='k')
        ax2.set_xlim(x_min, x_max)
        ax2.set_ylim(y_min, y_max)
        ax2.set_title('OvR 组合决策平面')
        plt.show()

### 3.2 一对一 (One-vs-One, OvO)

**核心思想**：将 $K$ 个类别两两配对组合，训练 $\frac{K(K-1)}{2}$ 个二分类模型。比如在三类(0,1,2)中存在三个组合：
- 模型一 (**0 vs 1**)：剔除其它项杂音，只拿类别 0 和 1 的混合数据作对战训练。
- 模型二 (**0 vs 2**)：只拿类别 0 和 2 的混合数据训练。
- 模型三 (**1 vs 2**)：只拿类别 1 和 2 的混合数据训练。

**预测机制（投票制）**：每个模型只能在它的执政范畴里给自己的类别投票加一分。全量模型投完票后：**得票总数最高的类别获胜**。

In [ ]:
pairs = [(0, 1), (0, 2), (1, 2)]
models = []

print("🚀 正在流水线式交替训练 3 套模型组合...")
for class_a, class_b in pairs:
    # 筛选只包含这个对战组合样本的“战区索引屏蔽”子集
    mask = (y_m_train_t == class_a) | (y_m_train_t == class_b)
    X_sub = X_m_train_t[mask]
    y_sub = y_m_train_t[mask]
    
    # 因为抽离为新的二分类战斗，在这里人为将 A 选做正方 (1)， B 做负方 (0)
    y_sub_bin = (y_sub == class_a).float().view(-1, 1)
    
    # 各有各的命门权重
    w_sub = torch.randn(2, 1, requires_grad=True)
    b_sub = torch.zeros(1, requires_grad=True)
    
    for _ in range(500):
        y_pred = torch.sigmoid(torch.mm(X_sub, w_sub) + b_sub)
        loss = -torch.mean(y_sub_bin * torch.log(y_pred + 1e-8) + (1 - y_sub_bin) * torch.log(1 - y_pred + 1e-8))
        loss.backward()
        with torch.no_grad():
            w_sub -= 0.5 * w_sub.grad
            b_sub -= 0.5 * b_sub.grad
            w_sub.grad.zero_()
            b_sub.grad.zero_()
            
    # 训练完毕，储存它的战场结晶参数
    models.append((class_a, class_b, w_sub.detach(), b_sub.detach()))
    print(f"✅ 模型战场 ({class_a} vs {class_b}) 训练完结。")

# --- 画布上的全民选举 ---
with torch.no_grad():
    votes = torch.zeros(grid_tensor.size(0), 3) # 为网格图层中千千万万的像素点设立3个类别的总票站
    for class_a, class_b, w_sub, b_sub in models:
        # 模型执行评估：得到判断这里是否属于 class_a 的信心概率
        prob_a = torch.sigmoid(torch.mm(grid_tensor, w_sub) + b_sub)
        
        # 将连续概率转化为两极反转判定，0.5是一个分水岭，投出一票！
        votes[:, class_a:class_a+1] += (prob_a >= 0.5).float()
        votes[:, class_b:class_b+1] += (prob_a < 0.5).float()
        
    Z_ovo = torch.argmax(votes, dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z_ovo, cmap=custom_cmap, alpha=0.3)
plt.scatter(X_m_train_t[:, 0], X_m_train_t[:, 1], c=y_m_train_t, cmap=custom_cmap, edgecolors='k')
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.title('OvO 全局投票决策地图')
plt.show()

### 3.3 Softmax (多项分布回归)

**核心思想**：我们放弃切割多个零散的二分类代理器，而是搭建一个直接推流成联合输出层的**全局网络**。
算出来的线性分值（常叫 Logits）通常跨度狂妄、参差不齐。我们把它送入特殊的 `Softmax` 归一池，通过指数膨胀和除权将其整理成**每一项介于 (0,1)间，且所有项相加和恒为 1 的严格概率分布表**。这是各大框架原生的最高频操作解法，与之配套的专供损失核称之为 **多分类交叉熵 (Cross Entropy)**。

In [ ]:
# 将映射关系由 2 个特征转换为对 3 种结果的独立得分（Logits分）权重
W_sm = torch.randn(2, 3, requires_grad=True)
b_sm = torch.zeros(3, requires_grad=True)

loss_history_sm = []

for epoch in range(epochs):
    # 1. 算出当前输入在各个类别的基础预选分 (shape: N个样本 x 3类得分)
    logits = torch.mm(X_m_train_t, W_sm) + b_sm
    
    # 2. **手动拆解 Softmax**: 计算 exp(x) / sum(exp(x))
    # 为防止浮点数在无穷逼近中越界，普遍做法会先将各项均相减最大值，不影响推导
    exp_logits = torch.exp(logits - torch.max(logits, dim=1, keepdim=True)[0]) 
    softmax_probs = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)
    
    # 3. **手动拆解 Cross Entropy**: 惩罚“我预测这道题是对的位置”和它的真正身份是否一致！
    # 取了对数后，从它 3 列宽的答案里精准点出真实标签那一列抽走该有的罚分 (Negative Log Likelihood)
    log_probs = torch.log(softmax_probs + 1e-8)
    batch_size = X_m_train_t.size(0)
    loss = -torch.sum(log_probs[range(batch_size), y_m_train_t]) / batch_size
    
    loss.backward()
    with torch.no_grad():
        W_sm -= 0.5 * W_sm.grad
        b_sm -= 0.5 * b_sm.grad
        W_sm.grad.zero_()
        b_sm.grad.zero_()
        
    loss_history_sm.append(loss.item())

    if (epoch + 1) % 50 == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.plot(loss_history_sm, 'g-')
        ax1.set_title(f'Softmax 多项联合交叉熵损失 (Epoch {epoch+1})')
        
        with torch.no_grad():
            grid_logits = torch.mm(grid_tensor, W_sm) + b_sm
            Z = torch.argmax(grid_logits, dim=1).numpy().reshape(xx.shape)
            
        ax2.contourf(xx, yy, Z, cmap=custom_cmap, alpha=0.3)
        ax2.scatter(X_m_train_t[:, 0], X_m_train_t[:, 1], c=y_m_train_t, cmap=custom_cmap, edgecolors='k')
        ax2.set_xlim(x_min, x_max)
        ax2.set_ylim(y_min, y_max)
        ax2.set_title(f'Softmax 全局推断决策地图')
        plt.show()

## 🌟 宏观总结

无论是将多分类切分（OvR/OvO），还是建立联合概率空间（Softmax），它们的边界由于只由线性方程和简单分布构成，生成的分类决策地带永远是**多条直线段拼合而成的线性边界**。这就是基于线性模型的天然属性。

下表归纳总结了这三种主流策略的对比差异：

| 对比维度 | 一对多 (OvR) | 一对一 (OvO) | Softmax (多项分布) |
| :--- | :--- | :--- | :--- |
| **底层实现依赖** | 利用 $K$ 个二分类组合 | 利用 $\frac{K(K-1)}{2}$ 个二分类组合 | **仅依靠 1 个**全局联合输出网络 |
| **预测（推理）机制** | 各模型自评，谁概率高听谁的 | 各模型互评投票，得票多者胜 | 直接取联合输出概率最大项 |
| **优缺点** | 思路框架扁平，但空间中可能存在盲区 | 拆分的对决极度纯粹精细，但类别越多训练成负荷灾害 | **当下神经网络分类算法的绝对基石**，理论自洽、求导平滑 |

> ✨ **前沿技术启示**：在如今的框架级（如 PyTorch / TensorFlow）业务开发中，一旦涉及处理多类别标签选项的任务，我们几乎全部并**唯一采用 Softmax 激活搭配 `CrossEntropyLoss` 核函数的方案**。之所以从头编写 OvR 与 OvO 的底层逻辑运算，意在让学习者能够切身看到从“最基础的是非题（二分类）”如何历经千锤百炼不断演进工程技术，最终汇聚并沉淀出了 Softmax —— 这一目前在主流多选问题中最通用、最优雅的算法核心。